# 🐒 Global Primates Watch — Limpeza e Preparação de Dados

Este notebook realiza o pipeline completo de limpeza de dados:
- Carregamento do shapefile IUCN
- Detecção automática de colunas
- Normalização de texto
- Filtragem de primatas
- Deduplicação por espécie
- Tradução de categorias IUCN para português
- Classificação de níveis de risco
- Exportação para formatos padrão (CSV, GeoJSON)

## 📦 Importações e Configuração

In [1]:
import sys
from pathlib import Path

# Adicionar diretório src ao path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root / 'src'))

import geopandas as gpd
import pandas as pd
import numpy as np
from data_utils import (
    load_shapefile,
    find_column,
    normalize_text_columns,
    filter_primates,
    deduplicate_species,
    translate_categories,
    classify_risk_level,
    export_to_csv,
    export_to_geojson
)

print("✓ Importações concluídas com sucesso!")

✓ Importações concluídas com sucesso!


## 📥 Carregar Shapefile IUCN

In [2]:
# Caminho para o shapefile
shapefile_path = project_root / 'data' / 'raw' / 'MAMMALS_TERRESTRIAL_ONLY' / 'MAMMALS_TERRESTRIAL_ONLY.shp'

print(f"Carregando shapefile: {shapefile_path}")
gdf = load_shapefile(str(shapefile_path))

print(f"\n✓ Shapefile carregado com sucesso!")
print(f"  - Linhas (polígonos): {len(gdf)}")
print(f"  - Colunas: {len(gdf.columns)}")
print(f"\nColunas disponíveis:")
print(gdf.columns.tolist())

Carregando shapefile: c:\Users\eferro\Desktop\Global Primates Watch\data\raw\MAMMALS_TERRESTRIAL_ONLY\MAMMALS_TERRESTRIAL_ONLY.shp

✓ Shapefile carregado com sucesso!
  - Linhas (polígonos): 12703
  - Colunas: 29

Colunas disponíveis:
['id_no', 'sci_name', 'presence', 'origin', 'seasonal', 'compiler', 'yrcompiled', 'citation', 'subspecies', 'subpop', 'source', 'island', 'tax_comm', 'dist_comm', 'generalisd', 'legend', 'kingdom', 'phylum', 'class', 'order_', 'family', 'genus', 'category', 'marine', 'terrestria', 'freshwater', 'SHAPE_Leng', 'SHAPE_Area', 'geometry']


## 🔍 Detectar e Validar Colunas

In [11]:
# Detectar nomes de colunas automaticamente
order_col = find_column(gdf.columns, ['order']) or 'order_'
cat_col = find_column(gdf.columns, ['category', 'cat']) or 'category'
name_col = find_column(gdf.columns, ['sci_name', 'binomial', 'name']) or 'sci_name'

print("Colunas detectadas:")
print(f"  - Ordem taxonômica: {order_col}")
print(f"  - Categoria IUCN: {cat_col}")
print(f"  - Nome científico: {name_col}")

# Verificar valores únicos
print(f"\nValores únicos em '{order_col}':")
print(gdf[order_col].unique()[:10])

print(f"\nValores únicos em '{cat_col}':")
print(gdf[cat_col].unique())

Colunas detectadas:
  - Ordem taxonômica: order_
  - Categoria IUCN: category
  - Nome científico: sci_name

Valores únicos em 'order_':
<ArrowStringArray>
[      'RODENTIA',     'CHIROPTERA',      'CARNIVORA',   'ARTIODACTYLA',
       'PRIMATES',   'AFROSORICIDA', 'DASYUROMORPHIA',   'EULIPOTYPHLA',
  'DIPROTODONTIA',     'LAGOMORPHA']
Length: 10, dtype: str

Valores únicos em 'category':
<ArrowStringArray>
['CR', 'VU', 'EN', 'LC', 'DD', 'NT', 'EX']
Length: 7, dtype: str


## 🧹 Normalizar Dados de Texto

In [4]:
# Normalizar colunas de texto
gdf = normalize_text_columns(gdf, [order_col, cat_col, name_col])

print("✓ Texto normalizado (maiúsculas, espaços removidos)")
print(f"\nAmostra de dados após normalização:")
print(gdf[[order_col, cat_col, name_col]].head(10))

✓ Texto normalizado (maiúsculas, espaços removidos)

Amostra de dados após normalização:
       order_ category              sci_name
0    RODENTIA       CR  ABROCOMA BOLIVIENSIS
1  CHIROPTERA       VU   ACERODON CELEBENSIS
2  CHIROPTERA       VU   ACERODON CELEBENSIS
3  CHIROPTERA       VU   ACERODON CELEBENSIS
4  CHIROPTERA       VU   ACERODON CELEBENSIS
5  CHIROPTERA       VU   ACERODON CELEBENSIS
6  CHIROPTERA       VU   ACERODON CELEBENSIS
7  CHIROPTERA       VU   ACERODON CELEBENSIS
8  CHIROPTERA       EN      ACERODON HUMILIS
9  CHIROPTERA       EN      ACERODON JUBATUS


## 🐒 Filtrar Primatas

In [5]:
# Filtrar apenas primatas
primates = filter_primates(gdf, order_col)

print(f"✓ Primatas filtrados com sucesso!")
print(f"  - Total de polígonos: {len(primates)}")
print(f"  - Percentual do total: {len(primates) / len(gdf) * 100:.1f}%")

print(f"\nDistribuição por categoria IUCN:")
print(primates[cat_col].value_counts().sort_index())

✓ Primatas filtrados com sucesso!
  - Total de polígonos: 922
  - Percentual do total: 7.3%

Distribuição por categoria IUCN:
category
CR    134
DD     13
EN    280
EX      2
LC    206
NT     89
VU    198
Name: count, dtype: int64


## 🔄 Deduplicação por Espécie

In [6]:
# Antes da deduplicação
print(f"Antes da deduplicação:")
print(f"  - Polígonos: {len(primates)}")
print(f"  - Espécies únicas: {primates[name_col].nunique()}")

# Deduplicar
primates_unique = deduplicate_species(primates, name_col)

print(f"\nDepois da deduplicação:")
print(f"  - Polígonos: {len(primates_unique)}")
print(f"  - Espécies únicas: {primates_unique[name_col].nunique()}")
print(f"  - Polígonos removidos: {len(primates) - len(primates_unique)}")

Antes da deduplicação:
  - Polígonos: 922
  - Espécies únicas: 526

Depois da deduplicação:
  - Polígonos: 526
  - Espécies únicas: 526
  - Polígonos removidos: 396


## 🌍 Traduzir Categorias IUCN

In [7]:
# Traduzir categorias
primates_unique = translate_categories(primates_unique, cat_col)

print("✓ Categorias traduzidas para português!")
print(f"\nMapeamento de categorias:")
translation_sample = primates_unique[[cat_col, 'category_pt']].drop_duplicates().sort_values(cat_col)
print(translation_sample.to_string(index=False))

✓ Categorias traduzidas para português!

Mapeamento de categorias:
category            category_pt
      CR Criticamente em Perigo
      DD    Dados Insuficientes
      EN              Em Perigo
      EX                Extinto
      LC      Pouco Preocupante
      NT         Quase Ameaçado
      VU             Vulnerável


## ⚠️ Classificar Níveis de Risco

In [8]:
# Classificar risco
primates_unique = classify_risk_level(primates_unique, cat_col)

print("✓ Níveis de risco classificados!")
print(f"\nDistribuição por nível de risco:")
print(primates_unique['risco'].value_counts())

print(f"\nEspécies em risco crítico:")
critical = primates_unique[primates_unique['risco'] == 'Crítico'][[name_col, 'category_pt', 'risco']]
print(f"  - Total: {len(critical)} espécies")
print(critical.head(10).to_string(index=False))

✓ Níveis de risco classificados!

Distribuição por nível de risco:
risco
Alto Risco      234
Baixo Risco     162
Médio Risco     115
Desconhecido     13
Crítico           2
Name: count, dtype: int64

Espécies em risco crítico:
  - Total: 2 espécies
                sci_name category_pt   risco
     XENOTHRIX MCGREGORI     Extinto Crítico
PALAEOPROPITHECUS INGENS     Extinto Crítico


## 📊 Estatísticas Finais

In [9]:
print("=" * 60)
print("RESUMO DA LIMPEZA DE DADOS")
print("=" * 60)

print(f"\n📈 Estatísticas Gerais:")
print(f"  - Espécies de primatas: {len(primates_unique)}")
print(f"  - Polígonos de distribuição: {len(primates_unique)}")

print(f"\n⚠️ Distribuição por Risco:")
risk_dist = primates_unique['risco'].value_counts()
for risk, count in risk_dist.items():
    pct = count / len(primates_unique) * 100
    print(f"  - {risk}: {count} ({pct:.1f}%)")

print(f"\n📍 Dados geoespaciais:")
print(f"  - CRS: {primates_unique.crs}")
print(f"  - Tipo de geometria: {primates_unique.geometry.type.unique()}")

print(f"\n✓ Dados prontos para análise!")

RESUMO DA LIMPEZA DE DADOS

📈 Estatísticas Gerais:
  - Espécies de primatas: 526
  - Polígonos de distribuição: 526

⚠️ Distribuição por Risco:
  - Alto Risco: 234 (44.5%)
  - Baixo Risco: 162 (30.8%)
  - Médio Risco: 115 (21.9%)
  - Desconhecido: 13 (2.5%)
  - Crítico: 2 (0.4%)

📍 Dados geoespaciais:
  - CRS: EPSG:4326
  - Tipo de geometria: <ArrowStringArray>
['MultiPolygon', 'Polygon']
Length: 2, dtype: str

✓ Dados prontos para análise!


## 💾 Exportar Dados Processados

In [10]:
# Preparar dados para exportação
output_dir = project_root / 'data' / 'processed'
output_dir.mkdir(parents=True, exist_ok=True)

# Exportar como CSV (sem geometria para compatibilidade)
primates_df = primates_unique.copy()
primates_df['geometry_wkt'] = primates_df.geometry.to_wkt()
primates_df_export = primates_df.drop(columns=['geometry'])

csv_path = output_dir / 'primates_species_clean.csv'
export_to_csv(primates_df_export, str(csv_path))

# Exportar como GeoJSON
geojson_path = output_dir / 'primates_map.geojson'
export_to_geojson(primates_unique, str(geojson_path))

print(f"\n✓ Todos os arquivos exportados com sucesso!")
print(f"  - CSV: {csv_path}")
print(f"  - GeoJSON: {geojson_path}")

✓ Arquivo exportado: c:\Users\eferro\Desktop\Global Primates Watch\data\processed\primates_species_clean.csv
✓ Arquivo exportado: c:\Users\eferro\Desktop\Global Primates Watch\data\processed\primates_map.geojson

✓ Todos os arquivos exportados com sucesso!
  - CSV: c:\Users\eferro\Desktop\Global Primates Watch\data\processed\primates_species_clean.csv
  - GeoJSON: c:\Users\eferro\Desktop\Global Primates Watch\data\processed\primates_map.geojson
